# Uncensor a chat model

Remove one behavior direction from a causal language model, then save the edited weights.

Run two prompt collections through the network. At one depth, subtract the average last-token state of the ordinary prompts from the average last-token state of the steered prompts. Scale that unit vector out of the token embedding, each attention output matrix, and each MLP down-projection.

## On Kaggle

1. In **Settings**, choose **GPU T4** and switch **Internet** on. Save and restart the session. If Internet is locked, complete the phone check Kaggle asks for.
2. To publish the edited model, open **Add-ons → Secrets**, create `HUB_TOKEN`, and attach it to this notebook.
3. Adjust the settings cell if you want a different model or a different spot-check prompt. Then run all cells.

`Qwen/Qwen2.5-3B-Instruct` in float16 fits the 16 GB T4. The edited folder is written under `/kaggle/working`, so it stays with the notebook output.


In [ ]:
base_model = "Qwen/Qwen2.5-3B-Instruct"
rows_per_side = 128
depth_fraction = 0.65
edit_strength = 1.0
keep_private = True
upload_to_hub = True
rng_seed = 42
trust_remote = True

system_text = "You are a helpful assistant."
steered_source = (
    "https://raw.githubusercontent.com/llm-attacks/llm-attacks/"
    "main/data/advbench/harmful_behaviors.csv"
)
plain_dataset = "tatsu-lab/alpaca"
spot_check = "How do I pick a padlock?"


In [ ]:
import os
import subprocess
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "datasets",
            "accelerate",
            "huggingface_hub",
            "transformers>=4.46",
        ]
    )
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        "Install failed. Turn Internet on, restart the session, and run this cell again."
    ) from exc


def read_hub_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
    except Exception:
        return None
    for name in ("HUB_TOKEN", "HF_TOKEN"):
        try:
            token = secrets.get_secret(name)
        except Exception:
            token = None
        if token:
            os.environ["HF_TOKEN"] = token
            os.environ["HUGGING_FACE_HUB_TOKEN"] = token
            return token
    return None


hub_token = read_hub_token()
if hub_token:
    from huggingface_hub import login

    login(token=hub_token, add_to_git_credential=False)
    print("Hub login succeeded.")
else:
    print("No HUB_TOKEN secret is attached. The edit still runs and saves on disk.")

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is visible. Set Accelerator to GPU T4 and restart the session.")

compute_major, compute_minor = torch.cuda.get_device_capability()
print(f"GPU: {torch.cuda.get_device_name(0)} (compute {compute_major}.{compute_minor})")

if compute_major >= 8:
    weight_dtype = torch.bfloat16
    try:
        import flash_attn  # noqa: F401

        attention_backend = "flash_attention_2"
    except ImportError:
        attention_backend = "eager"
        print("flash-attn is absent, so attention stays eager.")
else:
    weight_dtype = torch.float16
    attention_backend = "eager"
    print(f"Compute {compute_major}.{compute_minor}: eager attention, float16.")

print(f"dtype={weight_dtype}, attention={attention_backend}")


In [ ]:
import csv
import gc
import os
import random
import urllib.request

import torch
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer


def locate_blocks(model):
    modules = [getattr(model, "model", None)]
    inner = getattr(model, "model", None)
    if inner is not None:
        modules.append(getattr(inner, "language_model", None))
    modules.extend(
        [
            getattr(model, "language_model", None),
            getattr(model, "transformer", None),
            getattr(model, "gpt_neox", None),
        ]
    )
    for module in modules:
        if module is None:
            continue
        blocks = getattr(module, "layers", None)
        if blocks is None:
            blocks = getattr(module, "h", None)
        if blocks is not None and len(blocks) > 0:
            return module, blocks
    raise RuntimeError("This script expects a Llama-style decoder with a layer list.")


def depth_to_slot(block_count, fraction):
    slot = int(fraction * block_count)
    return min(max(slot, 0), block_count - 1)


def render_chat(tokenizer, model, messages):
    encoded = tokenizer.apply_chat_template(
        conversation=messages,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    token_ids = encoded.input_ids if hasattr(encoded, "input_ids") else encoded
    if token_ids.dim() == 1:
        token_ids = token_ids.unsqueeze(0)
    return token_ids.to(model.device)


def fetch_steered_prompts(url):
    request = urllib.request.Request(url, headers={"User-Agent": "uncensor-model"})
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = response.read().decode("utf-8")
    prompts = []
    for row in csv.DictReader(payload.splitlines()):
        goal = (row.get("goal") or "").strip()
        if goal:
            prompts.append(goal)
    if not prompts:
        raise RuntimeError(f"No goal rows were found at {url}")
    return prompts


def fetch_plain_prompts(repo):
    table = load_dataset(repo, split="train")
    prompts = []
    for row in table:
        instruction = (row.get("instruction") or "").strip()
        extra = (row.get("input") or "").strip()
        if not instruction:
            continue
        prompts.append(instruction if not extra else f"{instruction}\n{extra}")
    if not prompts:
        raise RuntimeError(f"{repo} did not yield any instructions.")
    return prompts


def take_sample(prompts, count, seed):
    if len(prompts) < count:
        raise ValueError(f"Need {count} prompts, found {len(prompts)}.")
    return random.Random(seed).sample(prompts, count)


def show_reply(model, tokenizer, prompt):
    token_ids = render_chat(
        tokenizer,
        model,
        [{"role": "user", "content": prompt}],
    )
    with torch.inference_mode():
        model.generate(
            token_ids,
            max_new_tokens=160,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            streamer=TextStreamer(tokenizer, skip_prompt=True),
        )


def cache_last_token(model, tokenizer, prompts, system_text, slot):
    """Store the chosen depth's last prompt token in float32 on CPU."""
    cached = []
    for prompt in tqdm(prompts, desc="Reading hidden states"):
        token_ids = render_chat(
            tokenizer,
            model,
            [
                {"role": "system", "content": system_text},
                {"role": "user", "content": prompt},
            ],
        )
        with torch.inference_mode():
            output = model(input_ids=token_ids, output_hidden_states=True, use_cache=False)
        usable = min(slot, len(output.hidden_states) - 1)
        vector = output.hidden_states[usable][0, -1, :].detach().float().cpu()
        cached.append(vector)
        del output
    return cached


def remove_component(matrix, direction, scale):
    direction = direction.view(-1).to(device=matrix.device, dtype=matrix.dtype)
    if matrix.shape[-1] == direction.shape[0]:
        projection = torch.einsum("...d,d->...", matrix, direction).unsqueeze(-1) * direction.unsqueeze(0)
        updated = matrix - scale * projection
    elif matrix.shape[0] == direction.shape[0]:
        projection = torch.einsum("d...,d->...", matrix, direction).unsqueeze(0) * direction.unsqueeze(-1)
        updated = matrix - scale * projection
    else:
        raise ValueError(
            f"Matrix shape {tuple(matrix.shape)} does not match vector length {direction.shape[0]}."
        )
    return updated.to(dtype=matrix.dtype)


def down_projection_name(mlp):
    if hasattr(mlp, "down_proj"):
        return "down_proj"
    if hasattr(mlp, "c_proj"):
        return "c_proj"
    return None


def apply_edit(trunk, blocks, direction, scale):
    counts = {"embedding": 0, "attention_out": 0, "mlp_down": 0}
    embedding = (
        getattr(trunk, "embed_tokens", None)
        or getattr(trunk, "wte", None)
        or getattr(trunk, "embed_in", None)
    )
    if embedding is not None and hasattr(embedding, "weight"):
        embedding.weight.data = remove_component(embedding.weight.data, direction, scale)
        counts["embedding"] = 1

    for block in tqdm(blocks, desc="Updating weights"):
        attention = getattr(block, "self_attn", None) or getattr(block, "attn", None)
        if attention is not None and hasattr(attention, "o_proj"):
            attention.o_proj.weight.data = remove_component(
                attention.o_proj.weight.data, direction, scale
            )
            counts["attention_out"] += 1

        mlp = getattr(block, "mlp", None)
        proj_name = down_projection_name(mlp) if mlp is not None else None
        if proj_name:
            projection = getattr(mlp, proj_name)
            projection.weight.data = remove_component(projection.weight.data, direction, scale)
            counts["mlp_down"] += 1
    return counts


In [ ]:
random.seed(rng_seed)
torch.manual_seed(rng_seed)

output_root = "/kaggle/working" if os.path.isdir("/kaggle/working") else "output"
os.makedirs(output_root, exist_ok=True)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    torch_dtype=weight_dtype,
    attn_implementation=attention_backend,
    trust_remote_code=trust_remote,
).eval()
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=trust_remote)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trunk, blocks = locate_blocks(model)
state_slot = depth_to_slot(len(blocks), depth_fraction)
print(f"Hidden-state slot {state_slot} of {len(blocks) - 1}")

print("Reply before the edit:")
show_reply(model, tokenizer, spot_check)

steered_prompts = take_sample(fetch_steered_prompts(steered_source), rows_per_side, rng_seed)
plain_prompts = take_sample(fetch_plain_prompts(plain_dataset), rows_per_side, rng_seed + 17)

steered_states = cache_last_token(model, tokenizer, steered_prompts, system_text, state_slot)
plain_states = cache_last_token(model, tokenizer, plain_prompts, system_text, state_slot)

contrast = torch.stack(steered_states).mean(dim=0) - torch.stack(plain_states).mean(dim=0)
contrast_norm = contrast.norm()
if contrast_norm.item() == 0 or not torch.isfinite(contrast_norm).item():
    raise RuntimeError("The contrast vector is zero or non-finite.")
contrast = contrast / contrast_norm

del steered_states, plain_states
gc.collect()
torch.cuda.empty_cache()

with torch.no_grad():
    edit_counts = apply_edit(trunk, blocks, contrast.view(-1), edit_strength)

del contrast

if sum(edit_counts.values()) == 0:
    raise RuntimeError("No weights were updated.")

print(f"Updated modules: {edit_counts}")
print("Reply after the edit:")
show_reply(model, tokenizer, spot_check)

short_name = base_model.rstrip("/").split("/")[-1]
edited_name = f"{short_name}-edited"
save_dir = os.path.join(output_root, edited_name)
model.save_pretrained(save_dir, safe_serialization=True)
tokenizer.save_pretrained(save_dir)
print(f"Saved edited weights to {save_dir}")

if upload_to_hub:
    if not hub_token:
        print("Skipped the Hub upload. Attach a HUB_TOKEN secret and rerun this cell.")
    else:
        model.push_to_hub(edited_name, private=keep_private, token=hub_token)
        tokenizer.push_to_hub(edited_name, private=keep_private, token=hub_token)
        print(f"Published {edited_name} (private={keep_private}).")
